In [29]:
pip install torch numpy matplotlib

In [30]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

2.10.0+cu128
True
Tesla T4
cuda


In [31]:
import torch
import torch.nn as nn
from torch.nn import functional as F


In [32]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt


--2026-05-25 14:09:58--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.2’

input.txt.2         100%[===================>]   1.06M  --.-KB/s    in 0.04s   

2026-05-25 14:09:58 (30.0 MB/s) - ‘input.txt.2’ saved [1115394/1115394]



In [33]:
with open("input.txt","r", encoding="utf-8") as f:
  text=f.read()

print(text[::300])

F br
sehnremaohr  h eldtliiuUWuwe Caioepem Tr e bbotw
w:
a t
ghtiR doeooogc.e
ro:scnn.ild d aictan , hmeIaooIuieU li ,Urdatrek ahngwos Uu hlIw,ON Lt io BdnhtA bih h  tw Svgwadohdi
 hwr n ?t 
yer,.educ soOneFsirpctd eodsoiA efY Cy senohou
 ue Ooenntnee,euao iLw e'hvmessfaooRoeol uh Mortthtb rsttktmbldhm Irekt nn SAe eri:io:em .I,oN 
W ISut'h fuokhirloeFAoc Id
 fuhtgr dlke.drm : naBi c,urenu ,

nenoo
wo ait
suUux,hhaoc .. rwoI i',tkere
ge
ab tcn 
eeN magnh rtsOmtoeTa acaashengsi
tda'VehsoBodbIs'hlo
  sbnt ,psrBmott sain neGeI,use!bTS lor ueaAa'ecuTcC tu Ye tRNo  nTnWilel ao IhvoeeeIanlwnMsTO Om:rwnZai  nnsvhdlvU
oMua neniars  cFs h
rdthl' auf C
h 
olparpre hg sHe ana::r o:,Eu spU oaboIt f broTce si   tGaaC   h:
 ow ln ttpecta  gsoos ouRGe  yrTarhcula?GiR   :in wyeHEGl,os Te t.ds 
oaomsslw
 l Osnh N,heroerh  t,h  hae hrG
lh
e eer  tzswe fggr
isuw onerMn nometttteb  Semc c ogm p u
 h
cIvts ne- n n,
hntnu oeo
o,Qlulhioo sU iu a 
thle. nomTyD Zo:eee
ssgTp  dkfja,un R Ho fi to ekIouryei utn i

In [34]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(chars)
print(vocab_size)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
65


In [35]:
stoi={ch:i for i, ch in enumerate(chars)}
itos={i:ch for i, ch in enumerate(chars)}
encode = lambda s:[stoi[c] for c in s]
decode = lambda l:''.join([itos[i] for i in l])
print(encode("hello"))
print(decode(encode(("hello"))))

[46, 43, 50, 50, 53]
hello


In [36]:
data = torch.tensor(encode(text),dtype=torch.long)
print(data.shape)
print(data[:100])

torch.Size([1115394])
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [37]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [38]:
block_size = 64
batch_size = 32
def get_batch(split):
  data = train_data if split == "train" else val_data
  ix = torch.randint(len(data)-block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])

  return x.to(device), y.to(device)

x, y = get_batch("train")

print(x)
print(y)

tensor([[ 0,  0, 17,  ..., 57,  1, 39],
        [61,  1, 51,  ..., 42, 43, 56],
        [43, 56, 43,  ..., 37, 53, 59],
        ...,
        [39, 58, 46,  ...,  1, 45, 56],
        [ 0, 23, 47,  ..., 58, 53,  1],
        [60, 43, 52,  ..., 46, 39, 58]], device='cuda:0')
tensor([[ 0, 17, 24,  ...,  1, 39,  1],
        [ 1, 51, 43,  ..., 43, 56,  8],
        [56, 43,  1,  ..., 53, 59,  1],
        ...,
        [58, 46,  5,  ..., 45, 56, 53],
        [23, 47, 52,  ..., 53,  1, 44],
        [43, 52, 47,  ..., 39, 58,  1]], device='cuda:0')


In [40]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embd):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
    def forward(self, idx, targets=None):
        token_embeddings = self.token_embedding_table(idx)
        logits = self.lm_head(token_embeddings)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

In [41]:
n_embd = 64
model = BigramLanguageModel(vocab_size, n_embd).to(device)

In [42]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
for steps in range(70000):
    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if steps % 100 == 0:
        print(loss.item())

4.326836585998535
3.0705244541168213
2.6692075729370117
2.5825648307800293
2.507011890411377
2.5177230834960938
2.515632152557373
2.505574941635132
2.484187364578247
2.4997594356536865
2.501668691635132
2.48600697517395
2.5264387130737305
2.4672365188598633
2.4837422370910645
2.483013153076172
2.4397430419921875
2.425097703933716
2.4506773948669434
2.465243339538574
2.4540486335754395
2.4279375076293945
2.4728033542633057
2.4663801193237305
2.482085704803467
2.4848647117614746
2.4699811935424805
2.5009641647338867
2.4127039909362793
2.4302940368652344
2.4825944900512695
2.459453582763672
2.433642864227295
2.473705768585205
2.4643702507019043
2.441723346710205
2.4285264015197754
2.4447741508483887
2.466020107269287
2.472604274749756
2.496229648590088
2.4970901012420654
2.4669580459594727
2.441709518432617
2.45070743560791
2.393552780151367
2.4530656337738037
2.463862419128418
2.4435548782348633
2.4201176166534424
2.4865612983703613
2.4957916736602783
2.438300132751465
2.4367194175720215

In [43]:
context = torch.zeros((1,1), dtype=torch.long, device=device)

generated_chars = model.generate(context, max_new_tokens=300)[0].tolist()

print(decode(generated_chars))


TI rspanonomigadspl ipherseA: brcler, t tha coun ave, st an u ghethee ain aband INot bumeal CHENo ty mad:
I

Masthommy, shancenth'Y hedisofrthrsthowe n s d:
RGUSoury thive d he lounhireang!
I tait btt merve y tordore dwinouke ghichive me myodstharo an st yos nth lkerant ansochary t t aceeapr er t he


In [44]:
!git config --global user.name "bhuvibhardwaj"
!git config --global user.email "bhuvibhardwaj999@gmail.com"

In [45]:
!git clone https://github.com/bhuvibhardwaj/MiniGPT.git

Cloning into 'MiniGPT'...
